# Node.js

Node.js is a free, open-source, cross-platform JavaScript runtime environment that allows developers to execute JavaScript code on the server side, outside of a web browser. Created by Ryan Dahl in 2009, it shifted JavaScript from a client-side scripting language into a powerful tool for full-stack web development.

## Core Architecture: How It Works

Node.js is not a programming language or a framework. It is a runtime built on two primary foundational components:

- **Google V8 Engine** — Written in C++, this engine compiles JavaScript directly into native machine code for ultra-fast execution.
- **Libuv Library** — A multi-platform C library that manages the system's operating system bindings. It handles the thread pool, file system events, and the core Event Loop.

### The Event Loop & Non-Blocking I/O

Traditional servers (like PHP or Apache) handle incoming requests by spawning a new thread for each connection. When a thread requests a database query, it blocks execution until the data returns.

Node.js operates on a **single-threaded event loop** model using asynchronous (non-blocking) I/O. Instead of waiting for a file read or API call to finish, Node.js delegates the task to the underlying operating system via Libuv and immediately moves to the next request. Once the task finishes, it triggers a callback function to resume execution. This permits a single server to manage thousands of concurrent connections efficiently without thread overhead.

> **Nuance:** "Single-threaded" describes *your JavaScript*, not the whole process. Libuv keeps a thread pool (4 threads by default, set via `UV_THREADPOOL_SIZE`) that handles file system operations, DNS lookups, and crypto work in the background.

### Event Loop Phases

Each iteration of the loop moves through fixed phases:

| Phase | What runs |
| --- | --- |
| **timers** | `setTimeout` and `setInterval` callbacks whose threshold has elapsed |
| **pending callbacks** | Deferred I/O callbacks from the previous cycle |
| **poll** | Retrieves new I/O events; executes most I/O callbacks |
| **check** | `setImmediate` callbacks |
| **close callbacks** | e.g. `socket.on('close', ...)` |

Between every phase, Node drains two microtask queues: `process.nextTick()` first, then resolved promise callbacks. This is why a `.then()` always fires before a `setTimeout(fn, 0)`.

## Key Architectural Differences

| Feature | Browser JavaScript (e.g. Chrome) | Node.js Runtime |
| --- | --- | --- |
| Global object | `window` | `global` (also `globalThis` in both) |
| DOM / BOM access | Yes (interacts with HTML, `document`) | No (cannot access UI elements) |
| System access | No (sandboxed for security) | Yes (direct access to files, OS, networks) |
| Module systems | ES Modules (`import`/`export`) | Natively supports both CommonJS (`require`) and ES Modules |
| Runtime-specific globals | `document`, `localStorage`, `alert` | `process`, `Buffer`, `__dirname`, `__filename` |

Note that a lot of what used to be browser-only is now shared: `fetch`, `URL`, `AbortController`, `TextEncoder`, `structuredClone`, `WebSocket`, and timers all exist in modern Node.

## Native Support & Language Extensions

- **JavaScript** — Fully executes modern ECMAScript standards with immediate runtime control over experimental features (via `--harmony` and other flags).
- **TypeScript** — Recent Node versions run `.ts` files directly by stripping type annotations, with no separate compile step. Caveat: it *erases* types rather than compiling them, so it does not type-check your code and does not support TypeScript-only constructs like `enum` or `namespace`. Keep `tsc --noEmit` in CI.
- **Core modules** — Built-in tools for low-level tasks, including `fs` (file system), `path` (path utilities), `http` (network servers), `os` (system telemetry), `crypto`, `stream`, `events`, `child_process`, and `worker_threads`. Import them with the `node:` prefix (`import fs from 'node:fs/promises'`) to make the origin explicit and avoid shadowing by npm packages.

## Modules: CommonJS vs. ESM

```javascript
// CommonJS — the historical default
const fs = require('node:fs');
module.exports = { myFunction };

// ES Modules — the modern standard
import fs from 'node:fs/promises';
export { myFunction };
```

Node decides which to use based on the file extension (`.cjs` vs `.mjs`) or the `"type"` field in `package.json`. Set `"type": "module"` for new projects. ESM files don't get `__dirname`; use `import.meta.dirname` instead.

## The npm Ecosystem

Node.js ships with npm, the Node Package Manager, hosting millions of open-source packages.

```bash
npm init -y                  # create package.json
npm install express          # add a runtime dependency
npm install -D vitest        # add a dev-only dependency
npm ci                       # clean, lockfile-exact install (use in CI)
npx <package>                # run a package without installing it globally
```

`package.json` records your dependencies and scripts; `package-lock.json` pins the exact resolved versions and **should be committed**. Alternatives to npm include pnpm (disk-efficient, strict) and Yarn.

## A Minimal Server

```javascript
import http from 'node:http';

const server = http.createServer((req, res) => {
  res.writeHead(200, { 'Content-Type': 'application/json' });
  res.end(JSON.stringify({ message: 'Hello from Node' }));
});

server.listen(3000, () => console.log('Listening on http://localhost:3000'));
```

In practice most people reach for a framework:

- **Express** — the long-standing default; minimal, huge middleware ecosystem.
- **Fastify** — faster, schema-based validation, modern plugin model.
- **NestJS** — opinionated and structured (decorators, DI), suits larger teams.
- **Hono / Elysia** — lightweight and edge-runtime friendly.

## Major Advantages

- **Single-language stack** — Write both front-end and back-end logic in JavaScript, maximizing code reuse and minimizing cognitive switching.
- **The npm ecosystem** — Millions of packages to rapidly accelerate development.
- **High scalability** — Ideal for I/O-intensive workloads, microservices, and lightweight APIs.
- **Real-time specialization** — The event-driven loop makes it a strong choice for chat systems, live collaboration tools, and gaming backends.
- **Fast startup** — Cheap to spin up, which suits containers and serverless functions.

## Ideal Use Cases vs. Limitations

### 🟢 When to Use Node.js

- **RESTful APIs & microservices** — Lightweight architecture makes it highly performant for routing data.
- **Real-time applications** — Excellent for continuous data streaming, WebSockets, and live chats.
- **Streaming services** — Native stream handling processes audio/video efficiently.
- **Build tooling & CLIs** — Most of the front-end toolchain (Vite, ESLint, Prettier) runs on Node.

### 🔴 When to Avoid Node.js

- **CPU-intensive tasks** — Heavy computation (image processing, video encoding, large-scale numeric work) blocks the single thread and stalls every other request. Mitigate with `worker_threads`, a native addon, or by offloading to a separate service — but if the *whole workload* is CPU-bound, Go, Rust, or Python may fit better.
- **Deeply CPU-bound scientific/ML work** — The library ecosystem in Python is simply far ahead here.

One correction worth making to the common version of this list: Node.js is **not** a poor fit for relational databases. Postgres and MySQL are extremely well served by mature drivers and ORMs (`pg`, Prisma, Drizzle, Knex). The "Node pairs with MongoDB" idea is a historical artifact of the MEAN stack, not a technical limitation — pick your database on data-modelling grounds, not runtime.

## Scaling Beyond One Core

A single Node process uses one CPU core for JavaScript. To use the rest:

- **`cluster`** — Forks multiple processes sharing one port, useful for handling more concurrent requests.
- **`worker_threads`** — Real threads within one process, sharing memory via `SharedArrayBuffer`. This is the right tool for CPU-bound work.
- **A process manager / orchestrator** — PM2, systemd, or Kubernetes replicas in production.

## Versions & Releases

Node follows a predictable cadence: a new major version every six months, and even-numbered releases move into Long Term Support with roughly 30 months of maintenance. **Always run an Active LTS version in production**; odd-numbered releases are short-lived and meant for testing new features. Use `nvm` (or `fnm`, or Volta) to manage multiple versions per project, and pin the version in `.nvmrc` and the `"engines"` field of `package.json`.

## Alternatives Worth Knowing

- **Deno** — Also from Ryan Dahl; secure by default, TypeScript-native, web-standard APIs.
- **Bun** — Written in Zig on JavaScriptCore; very fast, bundles a runtime, package manager, and test runner in one.

Both are largely Node-compatible, but Node remains the safe default for its ecosystem maturity and hosting support.